Install Required Libraries

In [ ]:
!pip install -q transformers==5.14.1 datasets==5.0.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.5 MB/s eta 0:00:00


Environment Check

In [ ]:
import torch
import transformers
import datasets
import sklearn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Scikit-learn:", sklearn.__version__)

print()

if torch.cuda.is_available():
    print("GPU Available:", True)
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU Available:", False)
    print("Running on CPU")

PyTorch: 2.11.0+cu128
Transformers: 5.14.1
Datasets: 5.0.1
Scikit-learn: 1.6.1

GPU Available: True
GPU: Tesla T4


Upload DAPT Model ZIP

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving roberta_dapt_model.zip to roberta_dapt_model.zip


Extract DAPT Model

In [ ]:
import zipfile
import os

zip_path = "roberta_dapt_model.zip"
extract_path = "./roberta_dapt_model"


with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)


print("DAPT model extracted successfully!")

print("\nFiles inside model folder:")
print(os.listdir(extract_path))

DAPT model extracted successfully!

Files inside model folder:
['config.json', 'tokenizer.json', 'model.safetensors', 'tokenizer_config.json']


Upload Final Classification Dataset

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Model final dataset.csv to Model final dataset.csv


Load Final Dataset

In [ ]:
import pandas as pd

dataset_path = "Model final dataset.csv"

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")

print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully!

Dataset Shape:
(15000, 2)

Columns:
['Sentences', 'Labels']

First 5 rows:


,Sentences,Labels
0,"We were missing a dwarf in the team, and now w...",0
1,useless guy,0
2,You are so rude!,0
3,Where does the goat live?,0
4,Why are you barking like a dog?,0


Shuffle and Split Dataset

In [ ]:
from sklearn.model_selection import train_test_split


df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Dataset shuffled successfully!")


train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Labels"]
)


val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["Labels"]
)


print("\nTraining samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))


print("\nTraining label distribution:")
print(train_df["Labels"].value_counts().sort_index())

print("\nValidation label distribution:")
print(val_df["Labels"].value_counts().sort_index())

print("\nTesting label distribution:")
print(test_df["Labels"].value_counts().sort_index())

Dataset shuffled successfully!

Training samples: 12000
Validation samples: 1500
Testing samples: 1500

Training label distribution:
Labels
0    2400
1    2400
2    2400
3    2400
4    2400
Name: count, dtype: int64

Validation label distribution:
Labels
0    300
1    300
2    300
3    300
4    300
Name: count, dtype: int64

Testing label distribution:
Labels
0    300
1    300
2    300
3    300
4    300
Name: count, dtype: int64


Load DAPT Tokenizer

In [ ]:
from transformers import RobertaTokenizer


model_path = "./roberta_dapt_model"


tokenizer = RobertaTokenizer.from_pretrained(model_path)


print("DAPT tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)
print("Maximum length:", tokenizer.model_max_length)

DAPT tokenizer loaded successfully!
Vocabulary size: 50265
Maximum length: 512


Convert Dataset to Hugging Face Format

In [ ]:
from datasets import Dataset


train_df = train_df.rename(columns={"Labels": "labels"})
val_df = val_df.rename(columns={"Labels": "labels"})
test_df = test_df.rename(columns={"Labels": "labels"})


train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)


print("Datasets created successfully!")

print("\nTraining:")
print(train_dataset)

print("\nValidation:")
print(val_dataset)

print("\nTesting:")
print(test_dataset)

Datasets created successfully!

Training:
Dataset({
    features: ['Sentences', 'labels', '__index_level_0__'],
    num_rows: 12000
})

Validation:
Dataset({
    features: ['Sentences', 'labels', '__index_level_0__'],
    num_rows: 1500
})

Testing:
Dataset({
    features: ['Sentences', 'labels', '__index_level_0__'],
    num_rows: 1500
})


Tokenize Dataset

In [ ]:
def clean_text(example):
    example["Sentences"] = str(example["Sentences"])
    return example


train_dataset = train_dataset.map(clean_text)
val_dataset = val_dataset.map(clean_text)
test_dataset = test_dataset.map(clean_text)


print("Text conversion completed!")
print(train_dataset[0])

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Text conversion completed!
{'Sentences': 'Why didnâ€™t anyone start impeachments on half of them yet', 'labels': 4, '__index_level_0__': 1677}


Tokenize Dataset (Retry)

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["Sentences"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)



train_dataset = train_dataset.remove_columns(
    ["Sentences", "__index_level_0__"]
)

val_dataset = val_dataset.remove_columns(
    ["Sentences", "__index_level_0__"]
)

test_dataset = test_dataset.remove_columns(
    ["Sentences", "__index_level_0__"]
)


print("Tokenization completed successfully!")

print(train_dataset[0])

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Tokenization completed successfully!
{'labels': 4, 'input_ids': [0, 7608, 399, 3695, 10851, 13445, 90, 1268, 386, 19828, 1488, 2963, 15, 457, 9, 106, 648, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


Load RoBERTa+DAPT Classification Model

In [ ]:
from transformers import RobertaForSequenceClassification


model = RobertaForSequenceClassification.from_pretrained(
    "./roberta_dapt_model",
    num_labels=5,
    ignore_mismatched_sizes=True
)


print("RoBERTa + DAPT classification model loaded successfully!")
print("Number of labels:", model.num_labels)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: ./roberta_dapt_model
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa + DAPT classification model loaded successfully!
Number of labels: 5


Set Dataset Format

In [ ]:
import torch

train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)


print("Dataset format set successfully!")

print("\nTraining columns:")
print(train_dataset.column_names)

print("\nValidation columns:")
print(val_dataset.column_names)

print("\nTesting columns:")
print(test_dataset.column_names)

Dataset format set successfully!

Training columns:
['labels', 'input_ids', 'attention_mask']

Validation columns:
['labels', 'input_ids', 'attention_mask']

Testing columns:
['labels', 'input_ids', 'attention_mask']


Create DataLoaders

In [ ]:
from torch.utils.data import DataLoader


batch_size = 16


train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size
)


print("DataLoaders created successfully!")

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

DataLoaders created successfully!
Training batches: 750
Validation batches: 94
Testing batches: 94


Training Setup

In [ ]:
import torch
from torch.optim import AdamW


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)


optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)


epochs = 3


print("Using device:", device)
print("Epochs:", epochs)
print("Learning rate:", 2e-5)
print("Optimizer: AdamW")

Using device: cuda
Epochs: 3
Learning rate: 2e-05
Optimizer: AdamW


Train RoBERTa + DAPT

In [ ]:
from tqdm.auto import tqdm


model.train()


for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    total_loss = 0


    progress_bar = tqdm(train_loader)


    for batch in progress_bar:

        optimizer.zero_grad()


        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)


        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )


        loss = outputs.loss


        loss.backward()

        optimizer.step()


        total_loss += loss.item()


        progress_bar.set_description(
            f"Loss: {loss.item():.4f}"
        )


    avg_loss = total_loss / len(train_loader)

    print(
        f"Average Training Loss: {avg_loss:.4f}"
    )


print("\nRoBERTa + DAPT training completed successfully!")


Epoch 1/3


  0%|          | 0/750 [00:00<?, ?it/s]

Average Training Loss: 0.2307

Epoch 2/3


  0%|          | 0/750 [00:00<?, ?it/s]

Average Training Loss: 0.0652

Epoch 3/3


  0%|          | 0/750 [00:00<?, ?it/s]

Average Training Loss: 0.0374

RoBERTa + DAPT training completed successfully!


Save RoBERTa + DAPT Fine-tuned Model

In [ ]:
save_path = "./roberta_dapt_finetuned_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


print("RoBERTa + DAPT model saved successfully!")
print("Saved location:", save_path)

import os
print("\nFiles:")
print(os.listdir(save_path))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa + DAPT model saved successfully!
Saved location: ./roberta_dapt_finetuned_model

Files:
['config.json', 'tokenizer.json', 'model.safetensors', 'tokenizer_config.json']


Zip and Download Model

In [ ]:
import shutil

folder = "roberta_dapt_finetuned_model"

zip_file = "roberta_dapt_finetuned_model"

shutil.make_archive(
    zip_file,
    'zip',
    folder
)

print("ZIP created successfully!")
print("File:", zip_file + ".zip")


from google.colab import files

files.download(zip_file + ".zip")

ZIP created successfully!
File: roberta_dapt_finetuned_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Load Test Dataset for Evaluation

In [ ]:
import torch

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

print("Test dataset ready!")

print(test_dataset[0])

Test dataset ready!
{'labels': tensor(1), 'input_ids': tensor([    0, 20328,     5,   276,  5617, 44267,    54,  2998,     5,    78,
        23069,    13,  3336,  1465,    33,     5,  4899, 32764,     7,  4744,
           10,  9468,    45,    95,   147,   171,   749,   115,  1029, 34632,
            6,    53,    67,    10,  9468,  9533,    30,   171,   215, 40752,
        14490,   116,     2,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
         

Load Fine-tuned RoBERTa + DAPT Model

In [ ]:
from transformers import RobertaForSequenceClassification, RobertaTokenizer
import torch


model_path = "./roberta_dapt_finetuned_model"


eval_model = RobertaForSequenceClassification.from_pretrained(
    model_path
)

eval_tokenizer = RobertaTokenizer.from_pretrained(
    model_path
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

eval_model.to(device)

eval_model.eval()


print("RoBERTa + DAPT evaluation model loaded successfully!")
print("Number of labels:", eval_model.num_labels)
print("Using device:", device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RoBERTa + DAPT evaluation model loaded successfully!
Number of labels: 5
Using device: cuda


Generate Predictions

In [ ]:
import torch
from tqdm.auto import tqdm


all_predictions = []
all_labels = []


eval_model.eval()


with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)


        outputs = eval_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )


        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )


        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )


print("Prediction completed!")

print("Total predictions:", len(all_predictions))
print("Total labels:", len(all_labels))

  0%|          | 0/94 [00:00<?, ?it/s]

Prediction completed!
Total predictions: 1500
Total labels: 1500


Calculate Evaluation Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions,
    average="weighted"
)

recall = recall_score(
    all_labels,
    all_predictions,
    average="weighted"
)

f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)


print("===== RoBERTa + DAPT Test Results =====")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")


print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions
    )
)

===== RoBERTa + DAPT Test Results =====
Accuracy : 0.9760
Precision: 0.9762
Recall   : 0.9760
F1 Score : 0.9760

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       300
           1       1.00      0.99      0.99       300
           2       0.95      0.96      0.96       300
           3       0.98      0.99      0.99       300
           4       0.98      0.97      0.98       300

    accuracy                           0.98      1500
   macro avg       0.98      0.98      0.98      1500
weighted avg       0.98      0.98      0.98      1500



Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd


cm = confusion_matrix(
    all_labels,
    all_predictions
)


cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {i}" for i in range(5)],
    columns=[f"Predicted {i}" for i in range(5)]
)


print("Confusion Matrix:")
display(cm_df)

Confusion Matrix:


,Predicted 0,Predicted 1,Predicted 2,Predicted 3,Predicted 4
Actual 0,290,0,8,2,0
Actual 1,1,296,2,1,0
Actual 2,5,0,289,1,5
Actual 3,2,0,0,298,0
Actual 4,2,0,6,1,291


EVALUATION

In [ ]:

from google.colab import files

uploaded = files.upload()

dapt_zip = list(uploaded.keys())[0]

print("DAPT model uploaded successfully!")
print("File:", dapt_zip)

Saving roberta_dapt_finetuned_model.zip to roberta_dapt_finetuned_model.zip
DAPT model uploaded successfully!
File: roberta_dapt_finetuned_model.zip


In [ ]:

import zipfile
import os

dapt_model_path = "./roberta_dapt_model_v2"

with zipfile.ZipFile(dapt_zip, "r") as zip_ref:
    zip_ref.extractall(dapt_model_path)

print("DAPT model extracted successfully!")
print("Location:", dapt_model_path)

print("\nFiles:")

for root, dirs, files_list in os.walk(dapt_model_path):
    for file in files_list:
        print(os.path.join(root, file))

DAPT model extracted successfully!
Location: ./roberta_dapt_model_v2

Files:
./roberta_dapt_model_v2/tokenizer.json
./roberta_dapt_model_v2/tokenizer_config.json
./roberta_dapt_model_v2/model.safetensors
./roberta_dapt_model_v2/config.json


In [ ]:
import pandas as pd
from google.colab import files

uploaded_dataset = files.upload()

dataset_filename = list(uploaded_dataset.keys())[0]

df = pd.read_csv(dataset_filename)

print("Dataset loaded successfully!")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

Saving Model final dataset.csv to Model final dataset.csv
Dataset loaded successfully!
Shape: (15000, 2)
Columns: ['Sentences', 'Labels']

First 5 rows:
                                           Sentences  Labels
0  We were missing a dwarf in the team, and now w...       0
1                                        useless guy       0
2                                   You are so rude!       0
3                          Where does the goat live?       0
4                    Why are you barking like a dog?       0


In [ ]:
from sklearn.model_selection import train_test_split

label_mapping = {
    0: "Insult",
    1: "Neutral",
    2: "Racism",
    3: "Sexual Abuse",
    4: "Threat"
}

train_df, temp_df = train_test_split(
    df,
    test_size=3000,
    stratify=df["Labels"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=1500,
    stratify=temp_df["Labels"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Datasets recreated successfully!")

print("\nTraining:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

print("\nTest distribution:")
print(test_df["Labels"].value_counts().sort_index())

print("\nLabel Mapping:")

for number, name in label_mapping.items():
    print(f"{number} -> {name}")

Datasets recreated successfully!

Training: (12000, 2)
Validation: (1500, 2)
Testing: (1500, 2)

Test distribution:
Labels
0    300
1    300
2    300
3    300
4    300
Name: count, dtype: int64

Label Mapping:
0 -> Insult
1 -> Neutral
2 -> Racism
3 -> Sexual Abuse
4 -> Threat


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

dapt_tokenizer = AutoTokenizer.from_pretrained(
    dapt_model_path
)

dapt_model = AutoModelForSequenceClassification.from_pretrained(
    dapt_model_path,
    num_labels=5
)

dapt_model.to(device)
dapt_model.eval()

print("\nRoBERTa+DAPT model loaded successfully!")
print("Number of labels:", dapt_model.config.num_labels)

Device: cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


RoBERTa+DAPT model loaded successfully!
Number of labels: 5


In [ ]:
test_texts = test_df["Sentences"].astype(str).tolist()
test_labels = test_df["Labels"].tolist()

dapt_test_encodings = dapt_tokenizer(
    test_texts,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

dapt_test_labels = torch.tensor(test_labels)

print("DAPT test dataset tokenized successfully!")

print(
    "Input IDs:",
    dapt_test_encodings["input_ids"].shape
)

print(
    "Attention Masks:",
    dapt_test_encodings["attention_mask"].shape
)

print(
    "Labels:",
    dapt_test_labels.shape
)

DAPT test dataset tokenized successfully!
Input IDs: torch.Size([1500, 128])
Attention Masks: torch.Size([1500, 128])
Labels: torch.Size([1500])


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

batch_size = 16

dapt_test_dataset = TensorDataset(
    dapt_test_encodings["input_ids"],
    dapt_test_encodings["attention_mask"],
    dapt_test_labels
)

dapt_test_loader = DataLoader(
    dapt_test_dataset,
    batch_size=batch_size,
    shuffle=False
)

dapt_predictions = []
dapt_probabilities = []
dapt_true_labels = []

with torch.no_grad():

    for input_ids, attention_mask, labels in dapt_test_loader:

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        outputs = dapt_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        probabilities = torch.softmax(logits, dim=1)

        predictions = torch.argmax(
            logits,
            dim=1
        )

        dapt_predictions.extend(
            predictions.cpu().numpy()
        )

        dapt_probabilities.extend(
            probabilities.cpu().numpy()
        )

        dapt_true_labels.extend(
            labels.numpy()
        )

print("DAPT prediction completed!")

print("Predictions:", len(dapt_predictions))
print("True labels:", len(dapt_true_labels))

DAPT prediction completed!
Predictions: 1500
True labels: 1500


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

dapt_accuracy = accuracy_score(
    dapt_true_labels,
    dapt_predictions
)

dapt_precision = precision_score(
    dapt_true_labels,
    dapt_predictions,
    average="weighted",
    zero_division=0
)

dapt_recall = recall_score(
    dapt_true_labels,
    dapt_predictions,
    average="weighted",
    zero_division=0
)

dapt_macro_f1 = f1_score(
    dapt_true_labels,
    dapt_predictions,
    average="macro",
    zero_division=0
)

dapt_weighted_f1 = f1_score(
    dapt_true_labels,
    dapt_predictions,
    average="weighted",
    zero_division=0
)

print("=" * 50)
print("RoBERTa + DAPT TEST RESULTS")
print("=" * 50)

print(f"Accuracy       : {dapt_accuracy:.4f}")
print(f"Precision      : {dapt_precision:.4f}")
print(f"Recall         : {dapt_recall:.4f}")
print(f"Macro F1       : {dapt_macro_f1:.4f}")
print(f"Weighted F1    : {dapt_weighted_f1:.4f}")

target_names = [
    "Insult",
    "Neutral",
    "Racism",
    "Sexual Abuse",
    "Threat"
]

print("\nClassification Report:")

print(
    classification_report(
        dapt_true_labels,
        dapt_predictions,
        target_names=target_names,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")

dapt_cm = confusion_matrix(
    dapt_true_labels,
    dapt_predictions
)

print(dapt_cm)

RoBERTa + DAPT TEST RESULTS
Accuracy       : 0.9953
Precision      : 0.9954
Recall         : 0.9953
Macro F1       : 0.9953
Weighted F1    : 0.9953

Classification Report:
              precision    recall  f1-score   support

      Insult     0.9934    0.9967    0.9950       300
     Neutral     1.0000    0.9967    0.9983       300
      Racism     0.9835    0.9933    0.9884       300
Sexual Abuse     1.0000    1.0000    1.0000       300
      Threat     1.0000    0.9900    0.9950       300

    accuracy                         0.9953      1500
   macro avg     0.9954    0.9953    0.9953      1500
weighted avg     0.9954    0.9953    0.9953      1500


Confusion Matrix:
[[299   0   1   0   0]
 [  0 299   1   0   0]
 [  2   0 298   0   0]
 [  0   0   0 300   0]
 [  0   0   3   0 297]]


In [ ]:
dapt_results = test_df.copy().reset_index(drop=True)

dapt_results["DAPT_Prediction"] = dapt_predictions

dapt_results["DAPT_Correct"] = (
    dapt_results["Labels"] ==
    dapt_results["DAPT_Prediction"]
)

dapt_probabilities_array = np.array(
    dapt_probabilities
)

for i, label_name in enumerate(target_names):

    dapt_results[
        f"DAPT_Prob_{label_name.replace(' ', '_')}"
    ] = dapt_probabilities_array[:, i]

dapt_results.to_csv(
    "roberta_dapt_test_predictions.csv",
    index=False
)

print("DAPT predictions saved successfully!")
print("File: roberta_dapt_test_predictions.csv")

print(
    "\nCorrect predictions:",
    dapt_results["DAPT_Correct"].sum()
)

print(
    "Incorrect predictions:",
    (~dapt_results["DAPT_Correct"]).sum()
)

DAPT predictions saved successfully!
File: roberta_dapt_test_predictions.csv

Correct predictions: 1493
Incorrect predictions: 7


In [ ]:
from google.colab import files

files.download(
    "roberta_dapt_test_predictions.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>